# First Step : Data Sourcing

### MeteoStats API

In [ ]:
import meteostat
print(meteostat.__version__)


1.7.6


In [ ]:
"""
extract_meteo_france.py
=======================
Extraction données météo HORAIRES sur 10 ans
pour les 96 départements de France métropolitaine
via Open-Meteo (gratuit, sans clé API, sans quota).

Installation :
    pip install openmeteo-requests requests-cache retry-requests tqdm pandas

Lancement dans Jupyter :
    exec(open('extract_meteo_france.py').read())
    main()
"""
import pandas as pd
import os
import time
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
from tqdm import tqdm
import openmeteo_requests
import requests_cache
from retry_requests import retry

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

START_DATE = "2015-01-01"
END_DATE   = "2024-12-31"

OUTPUT_DIR = Path("/home/berlh/code/stoak11/ceres_project/raw_data/openmeteo")
OUTPUT_DIR.mkdir(exist_ok=True)

CONSOLIDATED_FILE = OUTPUT_DIR / "meteo_france_horaire_10ans.csv"
LOG_FILE          = OUTPUT_DIR / "extraction.log"

SLEEP_BETWEEN_DEPTS = 0.5  # secondes entre chaque département

# Variables horaires à extraire
HOURLY_VARS = [
    "temperature_2m",           # Température (°C)
    "relative_humidity_2m",     # Humidité relative (%)
    "precipitation",            # Précipitations (mm)
    "snowfall",                 # Chutes de neige (cm)
    "wind_speed_10m",           # Vitesse vent (km/h)
    "wind_direction_10m",       # Direction vent (°)
    "wind_gusts_10m",           # Rafales (km/h)
    "surface_pressure",         # Pression surface (hPa)
    "sunshine_duration",        # Durée ensoleillement (s)
    "weather_code",             # Code météo WMO
    "dew_point_2m",             # Point de rosée (°C)
]

# ─────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# CLIENT OPEN-METEO (avec cache local)
# ─────────────────────────────────────────────

def build_client():
    cache_session = requests_cache.CachedSession(
        str(OUTPUT_DIR / ".om_cache"),
        expire_after=-1  # cache permanent
    )
    retry_session = retry(cache_session, retries=5, backoff_factor=0.5)
    return openmeteo_requests.Client(session=retry_session)

# ─────────────────────────────────────────────
# 96 DÉPARTEMENTS MÉTROPOLITAINS
# ─────────────────────────────────────────────

DEPARTEMENTS = {
    "01": {"nom": "Ain",                     "lat": 46.2044,  "lon":  5.2263},
    "02": {"nom": "Aisne",                   "lat": 49.5651,  "lon":  3.6233},
    "03": {"nom": "Allier",                  "lat": 46.5665,  "lon":  3.3654},
    "04": {"nom": "Alpes-de-Haute-Provence", "lat": 44.0922,  "lon":  6.2358},
    "05": {"nom": "Hautes-Alpes",            "lat": 44.5634,  "lon":  6.0785},
    "06": {"nom": "Alpes-Maritimes",         "lat": 43.7102,  "lon":  7.2620},
    "07": {"nom": "Ardèche",                 "lat": 44.7420,  "lon":  4.5952},
    "08": {"nom": "Ardennes",                "lat": 49.7718,  "lon":  4.7160},
    "09": {"nom": "Ariège",                  "lat": 42.9578,  "lon":  1.6066},
    "10": {"nom": "Aube",                    "lat": 48.2973,  "lon":  4.0744},
    "11": {"nom": "Aude",                    "lat": 43.2130,  "lon":  2.3491},
    "12": {"nom": "Aveyron",                 "lat": 44.3517,  "lon":  2.5756},
    "13": {"nom": "Bouches-du-Rhône",        "lat": 43.2965,  "lon":  5.3698},
    "14": {"nom": "Calvados",                "lat": 49.1829,  "lon": -0.3707},
    "15": {"nom": "Cantal",                  "lat": 45.0416,  "lon":  2.6494},
    "16": {"nom": "Charente",                "lat": 45.6497,  "lon":  0.1561},
    "17": {"nom": "Charente-Maritime",       "lat": 45.7469,  "lon": -0.6331},
    "18": {"nom": "Cher",                    "lat": 47.0810,  "lon":  2.3964},
    "19": {"nom": "Corrèze",                 "lat": 45.3481,  "lon":  1.8787},
    "21": {"nom": "Côte-d'Or",               "lat": 47.3220,  "lon":  5.0415},
    "22": {"nom": "Côtes-d'Armor",           "lat": 48.5148,  "lon": -2.7604},
    "23": {"nom": "Creuse",                  "lat": 46.1602,  "lon":  2.0236},
    "24": {"nom": "Dordogne",                "lat": 45.1844,  "lon":  0.7215},
    "25": {"nom": "Doubs",                   "lat": 47.2380,  "lon":  6.0227},
    "26": {"nom": "Drôme",                   "lat": 44.7333,  "lon":  4.8958},
    "27": {"nom": "Eure",                    "lat": 49.0236,  "lon":  1.1507},
    "28": {"nom": "Eure-et-Loir",            "lat": 48.4471,  "lon":  1.4888},
    "29": {"nom": "Finistère",               "lat": 48.0000,  "lon": -4.1000},
    "2A": {"nom": "Corse-du-Sud",            "lat": 41.9267,  "lon":  8.7369},
    "2B": {"nom": "Haute-Corse",             "lat": 42.3498,  "lon":  9.1497},
    "30": {"nom": "Gard",                    "lat": 43.8374,  "lon":  4.3601},
    "31": {"nom": "Haute-Garonne",           "lat": 43.6047,  "lon":  1.4442},
    "32": {"nom": "Gers",                    "lat": 43.6472,  "lon":  0.5854},
    "33": {"nom": "Gironde",                 "lat": 44.8378,  "lon": -0.5792},
    "34": {"nom": "Hérault",                 "lat": 43.6119,  "lon":  3.8772},
    "35": {"nom": "Ille-et-Vilaine",         "lat": 48.1173,  "lon": -1.6778},
    "36": {"nom": "Indre",                   "lat": 46.8166,  "lon":  1.6914},
    "37": {"nom": "Indre-et-Loire",          "lat": 47.3936,  "lon":  0.6898},
    "38": {"nom": "Isère",                   "lat": 45.1885,  "lon":  5.7245},
    "39": {"nom": "Jura",                    "lat": 46.6739,  "lon":  5.5549},
    "40": {"nom": "Landes",                  "lat": 43.8936,  "lon": -0.5001},
    "41": {"nom": "Loir-et-Cher",            "lat": 47.5935,  "lon":  1.3359},
    "42": {"nom": "Loire",                   "lat": 45.4333,  "lon":  4.3900},
    "43": {"nom": "Haute-Loire",             "lat": 45.0435,  "lon":  3.8851},
    "44": {"nom": "Loire-Atlantique",        "lat": 47.2184,  "lon": -1.5536},
    "45": {"nom": "Loiret",                  "lat": 47.9029,  "lon":  2.0026},
    "46": {"nom": "Lot",                     "lat": 44.4478,  "lon":  1.4413},
    "47": {"nom": "Lot-et-Garonne",          "lat": 44.3506,  "lon":  0.6208},
    "48": {"nom": "Lozère",                  "lat": 44.5196,  "lon":  3.4985},
    "49": {"nom": "Maine-et-Loire",          "lat": 47.4767,  "lon": -0.5540},
    "50": {"nom": "Manche",                  "lat": 49.1174,  "lon": -1.0776},
    "51": {"nom": "Marne",                   "lat": 49.0432,  "lon":  4.0237},
    "52": {"nom": "Haute-Marne",             "lat": 48.1078,  "lon":  5.1383},
    "53": {"nom": "Mayenne",                 "lat": 48.3010,  "lon": -0.6197},
    "54": {"nom": "Meurthe-et-Moselle",      "lat": 48.6921,  "lon":  6.1844},
    "55": {"nom": "Meuse",                   "lat": 49.1598,  "lon":  5.3828},
    "56": {"nom": "Morbihan",                "lat": 47.7497,  "lon": -2.8107},
    "57": {"nom": "Moselle",                 "lat": 49.1193,  "lon":  6.1757},
    "58": {"nom": "Nièvre",                  "lat": 47.1619,  "lon":  3.4965},
    "59": {"nom": "Nord",                    "lat": 50.6292,  "lon":  3.0573},
    "60": {"nom": "Oise",                    "lat": 49.4160,  "lon":  2.4267},
    "61": {"nom": "Orne",                    "lat": 48.4261,  "lon":  0.0916},
    "62": {"nom": "Pas-de-Calais",           "lat": 50.5199,  "lon":  2.6322},
    "63": {"nom": "Puy-de-Dôme",             "lat": 45.7796,  "lon":  3.0820},
    "64": {"nom": "Pyrénées-Atlantiques",    "lat": 43.2951,  "lon": -0.3708},
    "65": {"nom": "Hautes-Pyrénées",         "lat": 43.2330,  "lon":  0.0781},
    "66": {"nom": "Pyrénées-Orientales",     "lat": 42.6986,  "lon":  2.8956},
    "67": {"nom": "Bas-Rhin",                "lat": 48.5734,  "lon":  7.7521},
    "68": {"nom": "Haut-Rhin",               "lat": 47.7508,  "lon":  7.3359},
    "69": {"nom": "Rhône",                   "lat": 45.7640,  "lon":  4.8357},
    "70": {"nom": "Haute-Saône",             "lat": 47.6237,  "lon":  6.1551},
    "71": {"nom": "Saône-et-Loire",          "lat": 46.6667,  "lon":  4.8333},
    "72": {"nom": "Sarthe",                  "lat": 47.9955,  "lon":  0.1963},
    "73": {"nom": "Savoie",                  "lat": 45.5646,  "lon":  6.3914},
    "74": {"nom": "Haute-Savoie",            "lat": 45.8992,  "lon":  6.1294},
    "75": {"nom": "Paris",                   "lat": 48.8566,  "lon":  2.3522},
    "76": {"nom": "Seine-Maritime",          "lat": 49.4432,  "lon":  1.0993},
    "77": {"nom": "Seine-et-Marne",          "lat": 48.6000,  "lon":  2.8000},
    "78": {"nom": "Yvelines",                "lat": 48.8044,  "lon":  1.9863},
    "79": {"nom": "Deux-Sèvres",             "lat": 46.3236,  "lon": -0.4589},
    "80": {"nom": "Somme",                   "lat": 49.8941,  "lon":  2.2957},
    "81": {"nom": "Tarn",                    "lat": 43.7927,  "lon":  2.0790},
    "82": {"nom": "Tarn-et-Garonne",         "lat": 44.0167,  "lon":  1.3533},
    "83": {"nom": "Var",                     "lat": 43.1242,  "lon":  5.9280},
    "84": {"nom": "Vaucluse",                "lat": 43.9493,  "lon":  5.0480},
    "85": {"nom": "Vendée",                  "lat": 46.6709,  "lon": -1.4266},
    "86": {"nom": "Vienne",                  "lat": 46.5802,  "lon":  0.3404},
    "87": {"nom": "Haute-Vienne",            "lat": 45.8354,  "lon":  1.2617},
    "88": {"nom": "Vosges",                  "lat": 48.2200,  "lon":  6.4333},
    "89": {"nom": "Yonne",                   "lat": 47.7975,  "lon":  3.5674},
    "90": {"nom": "Territoire de Belfort",   "lat": 47.6384,  "lon":  6.8633},
    "91": {"nom": "Essonne",                 "lat": 48.6300,  "lon":  2.2200},
    "92": {"nom": "Hauts-de-Seine",          "lat": 48.8277,  "lon":  2.2396},
    "93": {"nom": "Seine-Saint-Denis",       "lat": 48.9356,  "lon":  2.3536},
    "94": {"nom": "Val-de-Marne",            "lat": 48.7754,  "lon":  2.4572},
    "95": {"nom": "Val-d'Oise",              "lat": 49.0500,  "lon":  2.1167},
}

# ─────────────────────────────────────────────
# EXTRACTION D'UN DÉPARTEMENT
# ─────────────────────────────────────────────

def extraire_departement(client, code: str, info: dict) -> pd.DataFrame | None:
    params = {
        "latitude":   info["lat"],
        "longitude":  info["lon"],
        "start_date": START_DATE,
        "end_date":   END_DATE,
        "hourly":     HOURLY_VARS,
        "timezone":   "Europe/Paris",
    }
    try:
        responses = client.weather_api(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params
        )
        r = responses[0]
        hourly = r.Hourly()

        dates = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ).tz_convert("Europe/Paris")

        df = pd.DataFrame({"datetime": dates})
        for i, var in enumerate(HOURLY_VARS):
            df[var] = hourly.Variables(i).ValuesAsNumpy()

        df.insert(0, "dept_code", code)
        df.insert(1, "dept_nom",  info["nom"])

        log.info(f"[{code}] {info['nom']} — {len(df):,} lignes extraites.")
        return df

    except Exception as e:
        log.error(f"[{code}] {info['nom']} — ERREUR : {e}")
        return None

# ─────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ─────────────────────────────────────────────

def main():
    log.info("=" * 60)
    log.info("DÉMARRAGE — Extraction météo horaire 10 ans (Open-Meteo)")
    log.info(f"Période    : {START_DATE} → {END_DATE}")
    log.info(f"Départements : {len(DEPARTEMENTS)}")
    log.info(f"Variables  : {HOURLY_VARS}")
    log.info("=" * 60)

    client     = build_client()
    all_frames = []
    erreurs    = []

    for code, info in tqdm(DEPARTEMENTS.items(), desc="Départements", unit="dept"):

        # Reprise automatique si déjà extrait
        dept_file = OUTPUT_DIR / f"dept_{code}.csv"
        if dept_file.exists():
            log.info(f"[{code}] Cache trouvé — chargement depuis {dept_file.name}")
            all_frames.append(pd.read_csv(dept_file))
            continue

        df = extraire_departement(client, code, info)

        if df is not None:
            df.to_csv(dept_file, index=False)
            all_frames.append(df)
        else:
            erreurs.append(code)

        time.sleep(SLEEP_BETWEEN_DEPTS)

    # ── Consolidation ──
    if all_frames:
        log.info("Consolidation en cours...")
        consolidated = pd.concat(all_frames, ignore_index=True)
        consolidated.to_csv(CONSOLIDATED_FILE, index=False)

        log.info("=" * 60)
        log.info(f"✅ TERMINÉ")
        log.info(f"   Lignes totales    : {len(consolidated):,}")
        log.info(f"   Départements OK   : {len(all_frames)}")
        log.info(f"   Erreurs           : {len(erreurs)} → {erreurs}")
        log.info(f"   Fichier consolidé : {CONSOLIDATED_FILE}")
        log.info("=" * 60)

        print("\n── Aperçu ──")
        print(consolidated.head(3).to_string())
        print(f"\nColonnes : {list(consolidated.columns)}")
        print(f"Shape    : {consolidated.shape}")
    else:
        log.error("Aucune donnée extraite.")


In [11]:
main()

2026-06-01 15:33:01,896 [INFO] ============================================================
2026-06-01 15:33:01,899 [INFO] DÉMARRAGE — Extraction météo horaire 10 ans (Open-Meteo)
2026-06-01 15:33:01,901 [INFO] Période    : 2015-01-01 → 2024-12-31
2026-06-01 15:33:01,902 [INFO] Départements : 96
2026-06-01 15:33:01,903 [INFO] Variables  : ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'surface_pressure', 'sunshine_duration', 'weather_code', 'dew_point_2m']
2026-06-01 15:33:01,905 [INFO] ============================================================
Départements:  15%|█▍        | 14/96 [00:03<00:33,  2.43dept/s]2026-06-01 15:33:05,322 [ERROR] [15] Cantal — ERREUR : failed to request 'https://archive-api.open-meteo.com/v1/archive': {'error': True, 'reason': 'Hourly API request limit exceeded. Please try again in the next hour.'}
2026-06-01 15:33:04,503 [ERROR] [16] Charente — ERREUR : failed to request 'http


── Aperçu ──
   dept_code dept_nom                   datetime  temperature_2m  relative_humidity_2m  precipitation  snowfall  wind_speed_10m  wind_direction_10m  wind_gusts_10m  surface_pressure  sunshine_duration  weather_code  dew_point_2m
0          1      Ain  2014-12-31 23:00:00+01:00         -2.2525             85.511665            0.0       0.0        7.517021           343.30070       16.919998         1004.5013                0.0           0.0       -4.3525
1          1      Ain  2015-01-01 00:00:00+01:00         -2.7525             86.428406            0.0       0.0        3.976330           354.80566       14.400000         1004.9323                0.0           0.0       -4.7025
2          1      Ain  2015-01-01 01:00:00+01:00         -1.0525             83.128140            0.0       0.0        3.706427           330.94550       10.799999         1005.5053                0.0           0.0       -3.5525

Colonnes : ['dept_code', 'dept_nom', 'datetime', 'temperature_2m', 'r

In [16]:
main()

2026-06-01 16:11:27,119 [INFO] ============================================================
2026-06-01 16:11:27,123 [INFO] DÉMARRAGE — Extraction météo horaire 10 ans (Open-Meteo)
2026-06-01 16:11:27,124 [INFO] Période    : 2015-01-01 → 2024-12-31
2026-06-01 16:11:27,126 [INFO] Départements : 96
2026-06-01 16:11:27,127 [INFO] Variables  : ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'surface_pressure', 'sunshine_duration', 'weather_code', 'dew_point_2m']
2026-06-01 16:11:27,128 [INFO] ============================================================
Départements:  95%|█████████▍| 91/96 [00:27<00:02,  1.86dept/s]2026-06-01 16:11:54,835 [ERROR] [91] Essonne — ERREUR : failed to request 'https://archive-api.open-meteo.com/v1/archive': {'reason': 'Daily API request limit exceeded. Please try again tomorrow.', 'error': True}
2026-06-01 16:11:53,689 [ERROR] [92] Hauts-de-Seine — ERREUR : failed to request 'https:


── Aperçu ──
  dept_code dept_nom                   datetime  temperature_2m  relative_humidity_2m  precipitation  snowfall  wind_speed_10m  wind_direction_10m  wind_gusts_10m  surface_pressure  sunshine_duration  weather_code  dew_point_2m
0         1      Ain  2014-12-31 23:00:00+01:00         -2.2525             85.511665            0.0       0.0        7.517021           343.30070       16.919998         1004.5013                0.0           0.0       -4.3525
1         1      Ain  2015-01-01 00:00:00+01:00         -2.7525             86.428406            0.0       0.0        3.976330           354.80566       14.400000         1004.9323                0.0           0.0       -4.7025
2         1      Ain  2015-01-01 01:00:00+01:00         -1.0525             83.128140            0.0       0.0        3.706427           330.94550       10.799999         1005.5053                0.0           0.0       -3.5525

Colonnes : ['dept_code', 'dept_nom', 'datetime', 'temperature_2m', 'relat

### METEO FRANCE

In [ ]:
# """
# download_meteofrance.py
# =======================
# Téléchargement des données météo HORAIRES 2010-2025
# pour les 96 départements de France métropolitaine
# depuis Météo-France / data.gouv.fr (gratuit, open data).

# Source : https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-horaires
# Licence : Licence Ouverte / Open Licence version 2.0

# Installation :
#     pip install requests tqdm pandas

# Lancement dans Jupyter :
#     exec(open('download_meteofrance.py').read())
#     main_meteofrance()

# Sortie :
#     meteofrance_horaire_2010_2025.csv  ← fichier consolidé final
#     raw/                               ← fichiers .csv.gz bruts
#     .done_XX                           ← marqueurs de reprise par département
# """

# import time
# import logging
# import requests
# from pathlib import Path

# import pandas as pd
# from tqdm import tqdm

# # ─────────────────────────────────────────────
# # CONFIGURATION
# # ─────────────────────────────────────────────

# ANNEE_DEBUT = 2010
# ANNEE_FIN   = 2025

# OUTPUT_DIR = Path("/home/berlh/code/stoak11/ceres_project/raw_data/meteofrance")
# RAW_DIR    = OUTPUT_DIR / "raw"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# RAW_DIR.mkdir(parents=True, exist_ok=True)

# CONSOLIDATED_FILE = OUTPUT_DIR / "meteofrance_horaire_2010_2025.csv"
# LOG_FILE          = OUTPUT_DIR / "download.log"

# BASE_URL = "https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/BASE/HOR"

# PERIODES_CIBLES = [
#     "2010-2019",
#     "previous-2020-2024",
#     "latest-2025-2026",
# ]

# # Colonnes à garder (20 sur 206) — divise la RAM par ~10
# COLONNES = [
#     "NUM_POSTE", "NOM_USUEL", "LAT", "LON", "ALTI",
#     "AAAAMMJJHH",
#     "T",     # Température (°C×10)
#     "U",     # Humidité relative (%)
#     "FF",    # Vent moyen 10m (m/s×10)
#     "DD",    # Direction vent (°)
#     "FXI",   # Rafale max (m/s×10)
#     "DXI",   # Direction rafale max (°)
#     "RR1",   # Précipitations 1h (mm×10)
#     "DRR1",  # Durée précipitations (min)
#     "NEIHE", # Neige fraîche (cm)
#     "PMER",  # Pression mer (hPa×10)
#     "GLO",   # Rayonnement global (J/cm²)
#     "INS",   # Insolation (min)
#     "VV",    # Visibilité (m)
#     "WW",    # Code temps présent WMO
# ]

# CHUNK_SIZE = 50_000  # lignes lues en mémoire à la fois
# SLEEP      = 0.3     # secondes entre requêtes

# # ─────────────────────────────────────────────
# # LOGGING
# # ─────────────────────────────────────────────

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s [%(levelname)s] %(message)s",
#     handlers=[
#         logging.FileHandler(LOG_FILE),
#         logging.StreamHandler()
#     ]
# )
# log = logging.getLogger(__name__)

# # ─────────────────────────────────────────────
# # 96 DÉPARTEMENTS MÉTROPOLITAINS
# # ─────────────────────────────────────────────

# DEPARTEMENTS = [
#     "01","02","03","04","05","06","07","08","09","10",
#     "11","12","13","14","15","16","17","18","19","21",
#     "22","23","24","25","26","27","28","29","2A","2B",
#     "30","31","32","33","34","35","36","37","38","39",
#     "40","41","42","43","44","45","46","47","48","49",
#     "50","51","52","53","54","55","56","57","58","59",
#     "60","61","62","63","64","65","66","67","68","69",
#     "70","71","72","73","74","75","76","77","78","79",
#     "80","81","82","83","84","85","86","87","88","89",
#     "90","91","92","93","94","95",
# ]

# # ─────────────────────────────────────────────
# # TÉLÉCHARGEMENT
# # ─────────────────────────────────────────────

# def build_url(dept: str, periode: str) -> str:
#     return f"{BASE_URL}/H_{dept}_{periode}.csv.gz"


# def telecharger_fichier(url: str, dest: Path) -> bool:
#     if dest.exists():
#         log.info(f"  Déjà présent : {dest.name}")
#         return True
#     try:
#         r = requests.get(url, timeout=120, stream=True)
#         if r.status_code == 404:
#             log.warning(f"  404 — inexistant : {url}")
#             return False
#         r.raise_for_status()
#         with open(dest, "wb") as f:
#             for chunk in r.iter_content(chunk_size=1024 * 1024):
#                 f.write(chunk)
#         log.info(f"  ✅ {dest.name} ({dest.stat().st_size / 1_048_576:.1f} Mo)")
#         return True
#     except Exception as e:
#         log.error(f"  ❌ {url} : {e}")
#         return False

# # ─────────────────────────────────────────────
# # TRAITEMENT PAR CHUNKS — écriture directe
# # ─────────────────────────────────────────────

# def traiter_departement(dept: str) -> bool:
#     """
#     Lit chaque fichier .csv.gz par chunks de CHUNK_SIZE lignes,
#     filtre sur la période cible et écrit directement dans le CSV final.
#     N'accumule jamais plus de ~50k lignes en RAM.
#     """
#     marker = OUTPUT_DIR / f".done_{dept}"
#     if marker.exists():
#         log.info(f"[{dept}] Déjà traité, skip.")
#         return True

#     for periode in PERIODES_CIBLES:
#         url  = build_url(dept, periode)
#         dest = RAW_DIR / f"H_{dept}_{periode}.csv.gz"

#         ok = telecharger_fichier(url, dest)
#         if not ok:
#             continue

#         try:
#             chunks = pd.read_csv(
#                 dest,
#                 sep=";",
#                 compression="gzip",
#                 dtype=str,
#                 low_memory=False,
#                 chunksize=CHUNK_SIZE,
#                 usecols=lambda c: c.strip() in COLONNES,
#             )

#             lignes_ecrites = 0
#             for chunk in chunks:
#                 chunk.columns = chunk.columns.str.strip()

#                 # Conversion date
#                 chunk["DATE"] = pd.to_datetime(
#                     chunk["AAAAMMJJHH"], format="%Y%m%d%H", errors="coerce"
#                 )
#                 chunk = chunk.dropna(subset=["DATE"])

#                 # Filtrage période
#                 chunk = chunk[
#                     (chunk["DATE"].dt.year >= ANNEE_DEBUT) &
#                     (chunk["DATE"].dt.year <= ANNEE_FIN)
#                 ]

#                 if chunk.empty:
#                     del chunk
#                     continue

#                 # Écriture directe — header seulement si fichier vide/inexistant
#                 write_header = not CONSOLIDATED_FILE.exists() or CONSOLIDATED_FILE.stat().st_size == 0
#                 chunk.to_csv(CONSOLIDATED_FILE, mode="a", index=False, header=write_header)
#                 lignes_ecrites += len(chunk)
#                 del chunk

#             log.info(f"[{dept}] {periode} — {lignes_ecrites:,} lignes écrites")

#         except Exception as e:
#             log.error(f"[{dept}] {periode} — ERREUR : {e}")

#         time.sleep(SLEEP)

#     # Marqueur de complétion pour la reprise
#     marker.touch()
#     log.info(f"[{dept}] ✅ Terminé")
#     return True

# # ─────────────────────────────────────────────
# # PIPELINE PRINCIPAL
# # ─────────────────────────────────────────────

# def main_meteofrance():
#     log.info("=" * 60)
#     log.info("DÉMARRAGE — Météo-France horaire 2010-2025")
#     log.info(f"Période      : {ANNEE_DEBUT} → {ANNEE_FIN}")
#     log.info(f"Départements : {len(DEPARTEMENTS)}")
#     log.info(f"Colonnes     : {len(COLONNES)}")
#     log.info(f"Chunk size   : {CHUNK_SIZE:,} lignes")
#     log.info(f"Sortie       : {CONSOLIDATED_FILE}")
#     log.info("=" * 60)

#     erreurs = []

#     for dept in tqdm(DEPARTEMENTS, desc="Départements", unit="dept"):
#         log.info(f"\n── Département {dept} ──")
#         ok = traiter_departement(dept)
#         if not ok:
#             erreurs.append(dept)

#     log.info("=" * 60)
#     log.info(f"✅ TERMINÉ")
#     log.info(f"   Erreurs      : {len(erreurs)} → {erreurs}")
#     log.info(f"   Fichier      : {CONSOLIDATED_FILE}")
#     if CONSOLIDATED_FILE.exists():
#         size_gb = CONSOLIDATED_FILE.stat().st_size / 1_073_741_824
#         log.info(f"   Taille       : {size_gb:.2f} Go")
#     log.info("=" * 60)

In [3]:
"""
download_meteofrance.py
=======================
Téléchargement des données météo HORAIRES 2010-2025
pour les 96 départements de France métropolitaine
depuis Météo-France / data.gouv.fr (gratuit, open data).

Source : https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-horaires
Licence : Licence Ouverte / Open Licence version 2.0

Installation :
    pip install requests tqdm pandas

Lancement dans Jupyter :
    exec(open('download_meteofrance.py').read())
    main_meteofrance()

Sortie :
    raw/           ← fichiers .csv.gz bruts (conservés pour reprise)
    dept_01.csv    ← données 2010-2025 complètes pour le département 01
    dept_02.csv    ← ...
    .done_01       ← marqueur de complétion (reprise automatique)
"""

import time
import logging
import requests
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

ANNEE_DEBUT = 2010
ANNEE_FIN   = 2026

OUTPUT_DIR = Path("/home/berlh/code/stoak11/ceres_project/raw_data/meteofrance")
RAW_DIR    = OUTPUT_DIR / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_DIR / "download.log"

BASE_URL = "https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/BASE/HOR"

PERIODES_CIBLES = [
    "2010-2019",
    "previous-2020-2024",
    "latest-2025-2026",
]

CHUNK_SIZE = 50_000  # lignes lues en mémoire à la fois
SLEEP      = 0.3     # secondes entre requêtes

# ─────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# 96 DÉPARTEMENTS MÉTROPOLITAINS
# ─────────────────────────────────────────────

DEPARTEMENTS = [
    "01","02","03","04","05","06","07","08","09","10",
    "11","12","13","14","15","16","17","18","19","21",
    "22","23","24","25","26","27","28","29","2A","2B",
    "30","31","32","33","34","35","36","37","38","39",
    "40","41","42","43","44","45","46","47","48","49",
    "50","51","52","53","54","55","56","57","58","59",
    "60","61","62","63","64","65","66","67","68","69",
    "70","71","72","73","74","75","76","77","78","79",
    "80","81","82","83","84","85","86","87","88","89",
    "90","91","92","93","94","95",
]

# ─────────────────────────────────────────────
# TÉLÉCHARGEMENT
# ─────────────────────────────────────────────

def build_url(dept: str, periode: str) -> str:
    return f"{BASE_URL}/H_{dept}_{periode}.csv.gz"


def telecharger_fichier(url: str, dest: Path) -> bool:
    if dest.exists():
        log.info(f"  Déjà présent : {dest.name}")
        return True
    try:
        r = requests.get(url, timeout=120, stream=True)
        if r.status_code == 404:
            log.warning(f"  404 — inexistant : {url}")
            return False
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        log.info(f"  ✅ {dest.name} ({dest.stat().st_size / 1_048_576:.1f} Mo)")
        return True
    except Exception as e:
        log.error(f"  ❌ {url} : {e}")
        return False

# ─────────────────────────────────────────────
# TRAITEMENT PAR CHUNKS — un CSV par département
# ─────────────────────────────────────────────

def traiter_departement(dept: str) -> bool:
    """
    Télécharge les fichiers .csv.gz d'un département,
    les lit chunk par chunk (50k lignes max en RAM),
    filtre sur 2010-2025 et écrit dans dept_XX.csv.
    Toutes les colonnes sont conservées.
    """
    marker    = OUTPUT_DIR / f".done_{dept}"
    dept_file = OUTPUT_DIR / f"dept_{dept}.csv"

    # Reprise automatique
    if marker.exists():
        log.info(f"[{dept}] Déjà traité, skip.")
        return True

    # Si dept_file existe mais pas le marqueur → crash partiel, on repart de zéro
    if dept_file.exists():
        dept_file.unlink()
        log.info(f"[{dept}] Fichier partiel supprimé, reprise.")

    first_write = True

    for periode in PERIODES_CIBLES:
        url  = build_url(dept, periode)
        dest = RAW_DIR / f"H_{dept}_{periode}.csv.gz"

        ok = telecharger_fichier(url, dest)
        if not ok:
            continue

        try:
            chunks = pd.read_csv(
                dest,
                sep=";",
                compression="gzip",
                dtype=str,
                low_memory=False,
                chunksize=CHUNK_SIZE,
            )

            lignes = 0
            for chunk in chunks:
                chunk.columns = chunk.columns.str.strip()

                # Conversion date (format YYYYMMDDHH)
                chunk["DATE"] = pd.to_datetime(
                    chunk["AAAAMMJJHH"], format="%Y%m%d%H", errors="coerce"
                )
                chunk = chunk.dropna(subset=["DATE"])

                # Filtrage période
                chunk = chunk[
                    (chunk["DATE"].dt.year >= ANNEE_DEBUT) &
                    (chunk["DATE"].dt.year <= ANNEE_FIN)
                ]

                if chunk.empty:
                    del chunk
                    continue

                chunk.to_csv(dept_file, mode="a", index=False, header=first_write)
                first_write = False
                lignes += len(chunk)
                del chunk

            log.info(f"[{dept}] {periode} — {lignes:,} lignes écrites")

        except Exception as e:
            log.error(f"[{dept}] {periode} — ERREUR : {e}")

        time.sleep(SLEEP)

    # Marqueur de complétion
    marker.touch()

    size_mb = dept_file.stat().st_size / 1_048_576 if dept_file.exists() else 0
    log.info(f"[{dept}] ✅ Terminé — {size_mb:.0f} Mo")
    return True

# ─────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ─────────────────────────────────────────────

def main_meteofrance():
    log.info("=" * 60)
    log.info("DÉMARRAGE — Météo-France horaire 2010-2025")
    log.info(f"Période      : {ANNEE_DEBUT} → {ANNEE_FIN}")
    log.info(f"Départements : {len(DEPARTEMENTS)}")
    log.info(f"Chunk size   : {CHUNK_SIZE:,} lignes")
    log.info(f"Sortie       : {OUTPUT_DIR}")
    log.info("=" * 60)

    erreurs = []

    for dept in tqdm(DEPARTEMENTS, desc="Départements", unit="dept"):
        log.info(f"\n── Département {dept} ──")
        ok = traiter_departement(dept)
        if not ok:
            erreurs.append(dept)

    # Résumé final
    dept_files = sorted(OUTPUT_DIR.glob("dept_*.csv"))
    total_go   = sum(f.stat().st_size for f in dept_files) / 1_073_741_824

    log.info("=" * 60)
    log.info(f"✅ TERMINÉ")
    log.info(f"   Fichiers créés : {len(dept_files)}")
    log.info(f"   Taille totale  : {total_go:.1f} Go")
    log.info(f"   Erreurs        : {len(erreurs)} → {erreurs}")
    log.info("=" * 60)

    print(f"\n✅ {len(dept_files)} départements — {total_go:.1f} Go total")
    print(f"Fichiers dans : {OUTPUT_DIR}")

In [4]:
main_meteofrance()

2026-06-01 23:26:12,595 [INFO] ============================================================
2026-06-01 23:26:12,599 [INFO] DÉMARRAGE — Météo-France horaire 2010-2025
2026-06-01 23:26:12,600 [INFO] Période      : 2010 → 2026
2026-06-01 23:26:12,601 [INFO] Départements : 96
2026-06-01 23:26:12,602 [INFO] Chunk size   : 50,000 lignes
2026-06-01 23:26:12,604 [INFO] Sortie       : /home/berlh/code/stoak11/ceres_project/raw_data/meteofrance
2026-06-01 23:26:12,606 [INFO] ============================================================
Départements:   0%|          | 0/96 [00:00<?, ?dept/s]2026-06-01 23:26:12,629 [INFO] 
── Département 01 ──
2026-06-01 23:26:13,706 [INFO]   ✅ H_01_2010-2019.csv.gz (35.4 Mo)
2026-06-01 23:27:04,858 [INFO] [01] 2010-2019 — 996,713 lignes écrites
2026-06-01 23:27:05,894 [INFO]   ✅ H_01_previous-2020-2024.csv.gz (20.4 Mo)
2026-06-01 23:27:39,834 [INFO] [01] previous-2020-2024 — 649,323 lignes écrites
2026-06-01 23:27:40,634 [INFO]   ✅ H_01_latest-2025-2026.csv.gz (5.4


✅ 94 départements — 70.3 Go total
Fichiers dans : /home/berlh/code/stoak11/ceres_project/raw_data/meteofrance


In [1]:
"""
consolider_meteofrance.py
=========================
Consolidation des 96 fichiers dept_XX.csv en un seul fichier
sans surcharger la RAM — lecture par chunks de 100k lignes.

Lancement dans Jupyter :
    exec(open('consolider_meteofrance.py').read())
    consolider()
"""

import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

OUTPUT_DIR        = Path("/home/berlh/code/stoak11/ceres_project/raw_data/meteofrance")
CONSOLIDATED_FILE = OUTPUT_DIR / "meteofrance_horaire_2010_2026.csv"
CHUNK_SIZE        = 100_000  # lignes lues en RAM à la fois

# ─────────────────────────────────────────────
# CONSOLIDATION
# ─────────────────────────────────────────────

def consolider():
    dept_files = sorted(OUTPUT_DIR.glob("dept_*.csv"))
    print(f"📂 {len(dept_files)} fichiers trouvés")

    if CONSOLIDATED_FILE.exists():
        CONSOLIDATED_FILE.unlink()
        print(f"🗑️  Ancien fichier consolidé supprimé")

    total_lignes = 0
    first_write  = True

    for i, f in enumerate(dept_files):
        dept_lignes = 0
        print(f"[{i+1:02d}/{len(dept_files)}] {f.name}...", end=" ", flush=True)

        try:
            for chunk in pd.read_csv(f, chunksize=CHUNK_SIZE, dtype=str, low_memory=False):
                chunk.to_csv(
                    CONSOLIDATED_FILE,
                    mode="a",
                    index=False,
                    header=first_write
                )
                first_write   = False
                dept_lignes  += len(chunk)
                total_lignes += len(chunk)
                del chunk

            size_mb = f.stat().st_size / 1_048_576
            print(f"{dept_lignes:,} lignes ({size_mb:.0f} Mo)")

        except Exception as e:
            print(f"❌ ERREUR : {e}")

    # Résumé
    size_gb = CONSOLIDATED_FILE.stat().st_size / 1_073_741_824
    print(f"\n✅ Terminé")
    print(f"   Lignes totales : {total_lignes:,}")
    print(f"   Taille fichier : {size_gb:.1f} Go")
    print(f"   Fichier        : {CONSOLIDATED_FILE}")

In [4]:
consolider()

📂 94 fichiers trouvés
[01/94] dept_01.csv... 1,831,219 lignes (614 Mo)
[02/94] dept_02.csv... 1,367,419 lignes (483 Mo)
[03/94] dept_03.csv... 2,924,301 lignes (927 Mo)
[04/94] dept_04.csv... 2,194,642 lignes (733 Mo)
[05/94] dept_05.csv... 2,928,986 lignes (998 Mo)
[06/94] dept_06.csv... 4,751,020 lignes (1599 Mo)
[07/94] dept_07.csv... 5,785,060 lignes (1829 Mo)
[08/94] dept_08.csv... 1,775,095 lignes (585 Mo)
[09/94] dept_09.csv... 1,313,310 lignes (446 Mo)
[10/94] dept_10.csv... 2,653,104 lignes (859 Mo)
[11/94] dept_11.csv... 2,818,393 lignes (997 Mo)
[12/94] dept_12.csv... 2,574,129 lignes (846 Mo)
[13/94] dept_13.csv... 2,709,673 lignes (971 Mo)
[14/94] dept_14.csv... 2,391,370 lignes (795 Mo)
[15/94] dept_15.csv... 2,269,908 lignes (742 Mo)
[16/94] dept_16.csv... 2,250,754 lignes (752 Mo)
[17/94] dept_17.csv... 2,436,323 lignes (826 Mo)
[18/94] dept_18.csv... 2,410,408 lignes (800 Mo)
[19/94] dept_19.csv... 2,639,962 lignes (852 Mo)
[20/94] dept_21.csv... 2,590,089 lignes (850 

In [3]:
from pathlib import Path

OUTPUT_DIR = Path("/home/berlh/code/stoak11/ceres_project/raw_data/meteofrance")

DEPARTEMENTS = [
    "01","02","03","04","05","06","07","08","09","10",
    "11","12","13","14","15","16","17","18","19","21",
    "22","23","24","25","26","27","28","29","2A","2B",
    "30","31","32","33","34","35","36","37","38","39",
    "40","41","42","43","44","45","46","47","48","49",
    "50","51","52","53","54","55","56","57","58","59",
    "60","61","62","63","64","65","66","67","68","69",
    "70","71","72","73","74","75","76","77","78","79",
    "80","81","82","83","84","85","86","87","88","89",
    "90","91","92","93","94","95",
]

presents  = {f.stem.replace("dept_", "") for f in OUTPUT_DIR.glob("dept_*.csv")}
manquants = [d for d in DEPARTEMENTS if d not in presents]
print(f"Présents  : {len(presents)}")
print(f"Manquants : {manquants}")

Présents  : 94
Manquants : ['2A', '2B']
